In [6]:
def compute_mean_gradients(mean_conn_matrix, display_output = True, data_reduction_algorithm = 'dm', kernel = 'cosine', save_screenshot = False, output_dir = None, sample_modality = None):
    
    '''
    
    Function that computes the mean gradients from mean connectivity matrix (across subjects)
    
    Input:
    - mean_conn_matrix: variable containing mean connectivity matrix across subjects
    - display_output: True (default) or False - displays the plots specified below
    - data_reduction_algorithm: used to compute the gradients. Options: 'dm' (diffusion map embedding; default), 'le' (laplacian eigenmaps), 'pca' (principal component analysis)
    - kernel: used to compute the gradietns. Default is cosine (for consistency with Margulies)
    - save_screenshot: True or False - if you want to save screenshot in output_dir. If True, the plots do not get displayed - so if want to save, need to run with True and then False to leave the plots visible in notebook
    - output_dir: path to output directory (e.g. resdir_fig, i.e. '/data/p_02667/sex_diff_gradients/HCP/results/')
    - sample_modality: string, e.g. GSP_local_ct or HCP_fc (for name of figures saved in output_dir)
    
    Output (display):
    - mean connectivity matrix + shape
    - 3 first mean connectivity gradients
    - scree plot of the variance explained by the 10 gradients + printed detail of variance explained and scaled varience explained
    
    Output:
    - mean_grad: the mean gradients, computed from the mean connectivity matrix with default parameters (diffusion map embedding, 10 gradients, threshold (fit -> sparsity = 0.90)
    
    
    '''  
    
    ## compute the mean gradients 
    
    # GradientMaps function used to build the model parameters
    mean_grad = GradientMaps(n_components = 10, random_state = 0, approach = data_reduction_algorithm, kernel = kernel)

    
    # fit function used to compute the gradients
    mean_grad.fit(mean_conn_matrix)
    
    
    if display_output:
        
        ## plot the mean connectivity matrix and shape
        
        plt.imshow(mean_conn_matrix)
        plt.show()
        print(mean_conn_matrix.shape)
        
        
        ## plot the 3 first mean gradients
        
        # defining labeling scheme and mask
        labeling = load_parcellation('schaefer', scale=400, join=True)
        surf_lh, surf_rh = load_conte69()
        mask = labeling != 0

        # list containing placeholders (None) for the number of gradients I want to plot
        grad = [None] * 3

        for i in range(3):
            # map the gradient to the parcels
            grad[i] = map_to_labels((mean_grad.gradients_)[:, i], labeling, mask=mask, fill=np.nan)  # mean_grad contains 10 .gradients_ (1 gradient per column) - here I take all rows and individual select column based on gradient I want (first 3)

        plot = plot_hemispheres(surf_lh, 
                                surf_rh, 
                                array_name=grad, 
                                embed_nb = True, 
                                size=(1200, 500),
                                cmap='viridis_r', 
                                nan_color = (0.7, 0.7, 0.7, 1),
                                color_bar=True, 
                                label_text=['Gradient 1', 'Gradient 2', 'Gradient 3'], 
                                zoom=1.8, # 1.55
                                screenshot = save_screenshot,
                                filename = output_dir+sample_modality+'_plotted_hemispheres_mean_gradients.png')
        
        display(plot)
        
        
        
        # Gradient 1 to save with the correct parameters
        mean_G1 = map_to_labels((mean_grad.gradients_)[:, 0], labeling, mask=mask, fill=np.nan)  # mean_grad contains 10 .gradients_ (1 gradient per column) - here I take all rows and individual select column based on gradient I want (first 3)

        plot_mean_G1 = plot_hemispheres(surf_lh, 
                                surf_rh, 
                                array_name=mean_G1, 
                                embed_nb = True, 
                                size= (1400,200), 
                                cmap='viridis_r', 
                                color_bar=True, 
                                nan_color = (0.7, 0.7, 0.7, 1),
                                label_text=['Mean G1'], 
                                zoom=1.45,
                                screenshot = save_screenshot,
                                filename = output_dir+sample_modality+'_plotted_hemispheres_mean_G1.png')
        

        ## plot the variance explained by the 10 gradients

        fig, ax = plt.subplots(1, figsize=(5, 4))
        ax.scatter(range(mean_grad.lambdas_.size), mean_grad.lambdas_)
        ax.set_title("Variance explained by the 10 gradients")
        ax.set_xlabel('Component Nb')
        ax.set_ylabel('Eigenvalue')
        plt.show()

        print(f"Total amount of variance explained by the {len(mean_grad.lambdas_)} gradients (uncorrected sum lambdas): {sum(mean_grad.lambdas_):.2f}\n")

        # Scaled variance explained by individual gradients: lambda / total(i.e., sum lambdas) * 100 %
        total = np.sum(mean_grad.lambdas_)
        scaled_var_explained_mean_grad = mean_grad.lambdas_ / total * 100
        print(f"G1: {scaled_var_explained_mean_grad[0]:.2f}%\nG2: {scaled_var_explained_mean_grad[1]:.2f}%\nG3: {scaled_var_explained_mean_grad[2]:.2f}")
    
    return mean_grad

In [4]:
def compute_aligned_gradients(conn_matrices, mean_grad_for_alignment, data_reduction_algorithm = 'dm', kernel = 'cosine'):
    
    '''
    
    Function that computes the alligned gradients from all subject connectivity matrices
    
    Input:
    - variable containing all subject connectivity matrices
    - array of the gradients to which the individual gradients should be aligned (the mean connectivity gradients)
    - data_reduction_algorithm: used to compute the gradients. Options: 'dm' (diffusion map embedding; default), 'le' (laplacian eigenmaps), 'pca' (principal component analysis)
    - kernel: used to compute the gradietns. Default is cosine (for consistency with Margulies)
    
    Output (dictionary containing arrays):  
    - array_aligned_gradients
    - array_aligned_G1
    - array_aligned_G2
    - array_aligned_G3
    
    Gradients computed from the mean connectivity matrix with default parameters (diffusion map embedding, 10 gradients, threshold (fit -> sparsity = 0.90)
    
    '''
    
    list_aligned_gradients = []  # will contain all participants all parcels (400), all gradients (10)
    list_aligned_G1 = []  # will contain all participants, all parcels (400) -> values = loadings for GRADIENT 1
    list_aligned_G2 = []  # will contain all participants, all parcels (400) -> values = loadings for GRADIENT 2
    list_aligned_G3 = []  # will contain all participants, all parcels (400) -> values = loadings for GRADIENT 3

    # loop over all the connectivity matrices
    for i in range(len(conn_matrices)):
        
        # setting model parameters for gradients to be computed across subjects - with procrustes alignment
        grad_procr = GradientMaps(n_components=10, random_state = 0, approach = data_reduction_algorithm, kernel = kernel, alignment='procrustes')  # specify alignment method here (procrustes)

        # computing
        # note that by using an alignment method, .fit yields a variable (grad_procr) containing both types of gradients, callable with: .gradients_ (original) and .aligned_ 
          # use ._gradients for mean_grad_for_alignment (mean grad was not even calculated with a reference so doesn't have ._aligned) and use .aligned_ for grad_procr 

        grad_procr.fit(conn_matrices[i], reference=mean_grad_for_alignment)  # align to the gradients of the gradients produced by the mean matrix (reference) 

        # append array to lists results (.T is necessary in order to be able to first access the gradients layer (10) so that can index the desired gradient, which will then contain all the parcels (400)
        list_aligned_gradients.append(grad_procr.aligned_)
        list_aligned_G1.append(grad_procr.aligned_.T[0])
        list_aligned_G2.append(grad_procr.aligned_.T[1])
        list_aligned_G3.append(grad_procr.aligned_.T[2])

    # make gradient lists into arrays (for analyses)    
    array_aligned_gradients = np.array(list_aligned_gradients)
    array_aligned_G1 = np.array(list_aligned_G1)
    array_aligned_G2 = np.array(list_aligned_G2)
    array_aligned_G3 = np.array(list_aligned_G3)
        
        
    ### dictionary to output
    
    dict_output = {'array_aligned_gradients': array_aligned_gradients, 'array_aligned_G1': array_aligned_G1, 'array_aligned_G2': array_aligned_G2, 'array_aligned_G3': array_aligned_G3}
    
    
    return dict_output

In [4]:
def compute_indvidual_gradients_NOT_aligned(conn_matrices, ref_SA_axis, data_reduction_algorithm = 'dm', kernel = 'cosine'):
    
    '''
    
    Function that computes the individual gradients from all subject connectivity matrices, NOT aligned to reference (mean) gradients, but still using a reference S-A axis to:
    - identify which gradient is S-A axis (highest absolute correlation to ref_SA_axis)
    - flip the identified S-A aixs gradient to face direction: sensory (-) to associaltion (+)
    
    Input:
    - variable containing all subject connectivity matrices
    - reference S-A axis (e.g., adult HCP)
    - data_reduction_algorithm: used to compute the gradients. Options: 'dm' (diffusion map embedding; default), 'le' (laplacian eigenmaps), 'pca' (principal component analysis)
    - kernel: used to compute the gradietns. Default is cosine (for consistency with Margulies)
    
    Output (dictionary containing arrays):  
    - array_aligned_gradients
    - array_SA_axis
    - array_grad_other1
    - array_grad_other2
    - SA_axis_grad_num  # number of gradient corresponding to S-A axis
    - scaled_var_exp_SA_axis # scaled varience explained by S-A axis
    - scaled_var_exp_other1  # scaled varience explained by other gradient 1
    - scaled_var_exp_other2  # scaled varience explained by other gradient 2
    - diff_scaled_var_exp_SA_axis_other1  # difference variance explained by S-A axis minus variance explained by other gradient 1
    
    Gradients computed from the with default parameters (diffusion map embedding, 10 gradients, threshold (fit -> sparsity = 0.90)
    
    '''

    
    list_SA_axis = []  # will contain all participants, all parcels (400) -> values = loadings for S-A axis
    list_SA_axis_grad_num = []  # will contain the number of the gradient corresponding to the S-A axis, per participant
    list_SA_axis_corr_to_ref = []  # will contain the correlation coefficient of S-A axis to adult gradient, per participant
    list_grad_other1 = []  # will contain all participants, all parcels (400) -> values = loadings for other gradient 1
    list_grad_other2 = []  # will contain all participants, all parcels (400) -> values = loadings for other gradient 2
    scaled_var_exp_SA_axis = []  # will contain all participants, the variance explained by S-A axis
    scaled_var_exp_other1 = []  # will contain all participants, the variance explained by other gradient 1
    scaled_var_exp_other2 = []  # will contain all participants, the variance explained by other gradient 2
    diff_scaled_var_exp_SA_axis_other1 = []  # will contain all participants, difference variance explained by S-A axis minus variance explained by other gradient 1

    

    # loop over all the connectivity matrices
    for i in range(len(conn_matrices)):
        
        ### setting model parameters for gradients to be computed across subjects - with no alignment
        grad = GradientMaps(n_components=10, random_state = 0, approach = data_reduction_algorithm, kernel = kernel)  

        
        ### computing gradients
          # note that when using an alignment method, .fit yields a variable (grad_procr) containing both types of gradients, callable with: .gradients_ (original) and .aligned_ 
            # use ._gradients for not aligned (if it's not computed with a reference for alignment, it won't even have a callable ._aligned) 
            # use .aligned_ for when procrustes alignment 
        grad.fit(conn_matrices[i])  

        
        ### identify which gradient is S-A axis and flip it if necessary to match direction sensory (-) to association (+)
          # .T is necessary in order to be able to first access the gradients layer (10) so that can index the desired gradient, which will then contain all the parcels (400)

        gradients = grad.gradients_.T


        # compute Spearman correlations with the reference array
        correlations = [stats.spearmanr(ref_SA_axis, arr)[0] for arr in gradients]
        abs_correlations = [abs(corr) for corr in correlations]
        
        # get indices of the top 3 gradients (by absolute correlation)
        top3_indices = np.argsort(abs_correlations)[-3:][::-1]  # sorted descending
        
        # first = S-A axis (best match)
        best_idx = top3_indices[0]
        best_array = gradients[best_idx]
        best_corr = correlations[best_idx]
        
        # flip sign if needed
        if best_corr < 0:
            best_array = -best_array
        
        # store S-A axis
        list_SA_axis.append(best_array)
        list_SA_axis_grad_num.append(best_idx + 1)
        list_SA_axis_corr_to_ref.append(abs(best_corr))
        
        # store other two gradients
        other_indices = top3_indices[1:]  # get the other two
        list_grad_other1.append(gradients[other_indices[0]])
        list_grad_other2.append(gradients[other_indices[1]])

    
        # compute scaled variance explained (list is in gradient order 1-10)
        sum_lambdas = np.sum(grad.lambdas_)
        scaled_var_exp_list = grad.lambdas_ / sum_lambdas * 100

        # store variance explained of S-A axis and other 2 gradients
        scaled_var_exp_SA_axis.append(scaled_var_exp_list[best_idx])
        scaled_var_exp_other1.append(scaled_var_exp_list[other_indices[0]])
        scaled_var_exp_other2.append(scaled_var_exp_list[other_indices[1]])

        # compute the difference between variance explained by S-A axis MINUS variance explained by other gradient 1 -> negative value if flip has not yet occured
        diff = scaled_var_exp_list[best_idx] - scaled_var_exp_list[other_indices[0]]
        diff_scaled_var_exp_SA_axis_other1.append(diff)


    # make gradient lists into arrays (for analyses)    
    array_SA_axis = np.array(list_SA_axis)
    array_grad_other1 = np.array(list_grad_other1)
    array_grad_other2 = np.array(list_grad_other2)
        
        
    ### dictionary to output
    
    dict_output = {'array_SA_axis': array_SA_axis, 'array_grad_other1': array_grad_other1, 'array_grad_other2': array_grad_other2, 'list_SA_axis_grad_num': list_SA_axis_grad_num, 'list_SA_axis_corr_to_ref': list_SA_axis_corr_to_ref, 'scaled_var_exp_SA_axis': scaled_var_exp_SA_axis, 'scaled_var_exp_other1': scaled_var_exp_other1, 'scaled_var_exp_other2': scaled_var_exp_other2, 'diff_scaled_var_exp_SA_axis_other1': diff_scaled_var_exp_SA_axis_other1}
    
    
    return dict_output

In [3]:
def fs5_to_schaefer400(fs5_data, seven_networks=True):
    
    '''
    Function that transforms data in fsaverage5 space to Schaefer 400 (7 or 17 networks) space.
    
    Input:
    - fs5_data: data in fsaverage5 space (array with 20484 vertices)
    - seven_networks: True (for 7 networks) or False (for 17 networks)
    
    Output:
    - data in Schaefer 400 space (array with 400 elements)
    
    '''
    
    # fetch the fsaverage parcellation labeling each of fsaverage5's 20484 vertices with its corresponding Schaefer parcel - seven_networks=False means 17 yeo networks
    from brainstat.datasets import fetch_parcellation
    
    schaefer_400_fs5 = fetch_parcellation("fsaverage5", "schaefer", 400, seven_networks=seven_networks)

    # Initialize array to hold Schaefer 400 data
    schaefer400_data = np.zeros(400)

    # Initialize array to keep track of the number of fs5 vertices within each Schaefer parcel
    parcel_counts = np.zeros(400)
    
    # Iterate over fs5 vertices
    for i, parcel in enumerate(schaefer_400_fs5):
        if parcel > 0:  # Ignore midline or non-parcel vertices (usually labeled as 0)
            schaefer400_data[parcel-1] += fs5_data[i]  # Accumulate fs5 values to the corresponding parcel
            parcel_counts[parcel-1] += 1  # Count number of vertices contributing to each parcel

    # Compute the average value for each Schaefer 400 parcel
    schaefer400_data /= parcel_counts
    
    # Replace divisions by zero with NaN (in case a parcel has no vertices assigned)
    schaefer400_data[parcel_counts == 0] = np.nan

    return schaefer400_data
